# 17 — Embedding design amplifies the effect

The capstone's win over the phase-locking value was real but *small* — about $0.004$ nats, a hair below the resolution of today's quantum hardware. This notebook turns a dial the capstone left at its default and shows the effect can be made **far larger and far more significant**, purely by choosing the embedding more deliberately. The advantage stays *quantum-inspired* — a better-organised classical analysis — but it becomes comfortably measurable.

**Prerequisites:** `15_discriminating_experiment` (the matched-PLV ensembles and the rich-QMI gap), `16_capstone_synthesis` (the honest accounting and the embedding-as-dial idea), `14_richer_embedding_applied` (the multi-frequency qutrit).

**What you'll be able to do**
- Use `weighted_multifreq_state` to set *how strongly* each circular-moment channel is encoded into $\rho_{AB}$.
- Show that **down-weighting the first-harmonic (PLV) channel** both enlarges the rich-QMI gap and collapses the null's spread — a denoising-by-design effect that lifts the separation by orders of magnitude in significance.
- See two further levers: adding harmonics, and the *amplitude* embedding, which opens a phase–amplitude coupling channel PLV cannot reach.
- State precisely what this buys — **measurability**, not exclusivity — keeping the honest accounting of `16` intact.

In `16` we read the capstone result with both eyes open: a controlled, significant separation, but a modest one, riding a single channel of the joint density. We left a question on the table — *was $0.004$ nats the best the framework can do, or only its default?* Here we answer it. The embedding is a dial, and we have barely turned it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import kl_divergence
from qot_course.quantum_ot.capstone import plv, coupling_qmi
from qot_course.quantum_ot.embeddings import (
    multifreq_state,
    weighted_multifreq_state,
    joint_density_from_states,
    amplitude_phase_state,
)
from qot_course.quantum_ot.discrimination import matched_plv_ensembles, sample_phase_difference

np.random.seed(42)
viz.use_course_style()

# Sample count per channel: large enough that the circular moments are tight Monte-Carlo
# estimates (error ~ 1/sqrt(N)), matching the discriminating experiment of notebook 15.
N = 150_000

## 1. The dial we left at its default

The multi-frequency qutrit of `14` is an *equal-weight* superposition: $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$. Each level carries the same amplitude $1/\sqrt 3$, so the joint density $\rho_{AB}$ spreads the coupling evenly across its channels: the first-moment coherence (PLV's channel) and the second-moment coherence (the discriminating one) sit side by side, equally weighted.

But the first-moment channel is exactly the one PLV already reads — and in the matched-PLV experiment, it is *matched* by construction, so it contributes **nothing but noise** to the separation. What if we turned it down? `weighted_multifreq_state` lets us set the per-channel weights $(w_{\mathrm{DC}}, w_{h_1}, w_{h_2})$; the coherence carrying the $h$-th moment scales like $w_{\mathrm{DC}}^2\,w_h^2$. Down-weighting $w_{h_1}$ suppresses the PLV channel and concentrates the measure on the second moment.

We rebuild the two matched-PLV ensembles from `15` and sweep the weighting.

In [ ]:
# The two ensembles from notebook 15: identical first moment a1 (-> identical PLV),
# different second moment a2 (0.0 vs 0.3). The richer measure must separate them.
(ta_lo, tb_lo), (ta_hi, tb_hi) = matched_plv_ensembles(
    N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=0
)
print(f"PLV gap (matched by construction): {abs(plv(ta_lo, tb_lo) - plv(ta_hi, tb_hi)):.4f}")


# Quantum mutual information (nats) of the weighted-qutrit joint embedding.
def rich_qmi(ta, tb, weights):
    rho = joint_density_from_states(
        weighted_multifreq_state(ta, (1, 2), weights),
        weighted_multifreq_state(tb, (1, 2), weights),
    )
    return coupling_qmi(rho, dims=(3, 3))


configs = [
    ("equal weights (1, 1, 1)", (1.0, 1.0, 1.0)),
    ("down-weight PLV (1, 0.5, 2)", (1.0, 0.5, 2.0)),
    ("down-weight PLV (1, 0.2, 2)", (1.0, 0.2, 2.0)),
]
gaps = []
for name, w in configs:
    g = abs(rich_qmi(ta_lo, tb_lo, w) - rich_qmi(ta_hi, tb_hi, w))
    gaps.append(g)
    print(f"  {name:30s} rich-QMI gap = {g:.5f} nats")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
y = np.arange(len(configs))[::-1]
bar_colors = [COLORS["muted"], COLORS["quantum"], COLORS["highlight"]]
ax.barh(y, gaps, color=bar_colors, edgecolor="white", linewidth=1.2, zorder=3)
# The plausible hardware noise floor (sigma_stat ~ 0.012 nats, notebook 05/06).
ax.axvline(0.012, color=COLORS["negative"], lw=1.6, ls="--", zorder=2,
           label="plausible hardware noise floor\n(~0.012 nats, 05/06)")
for yi, g in zip(y, gaps):
    ax.annotate(f"{g:.4f}", (g, yi), xytext=(6, 0), textcoords="offset points",
                va="center", ha="left", fontsize=10, color=COLORS["text"])
ax.set_yticks(y)
ax.set_yticklabels([name for name, _ in configs])
ax.set_xlabel("rich-QMI gap between the two matched-PLV ensembles  (nats)")
ax.set_xlim(0, max(gaps) * 1.3)
ax.set_title("Down-weighting the PLV channel enlarges the effect", pad=10)
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

**Read the figure.** Three weightings of the *same* qutrit embedding, applied to the *same* two ensembles. The grey bar is the equal-weight default from `15` — the familiar $\approx 0.004$ nat gap, sitting below the dashed hardware noise floor. As we turn the first-harmonic weight down (blue, then rose), the gap climbs: the rose bar, with the PLV channel cut to a fifth of its weight, lands around three times higher — and crosses the noise floor. Nothing about the *systems* changed; we stopped spending the measure's sensitivity on the channel that was matched anyway, and spent it on the one that discriminates. The dial was there all along.

## 2. The real prize: significance, not merely size

A larger gap is nice; a larger gap against a *quieter* null is decisive. Down-weighting the PLV channel does something stronger than enlarge the effect — it removes the dominant source of run-to-run noise. The first-moment channel is matched between the ensembles, so every bit of its finite-sample wobble feeds the null distribution. Cut that channel and the null nearly vanishes.

We measure it: the rich-QMI gap on an $a_2$-**matched** null (both ensembles $a_2 = 0$, so nothing to find) across several seeds, for the equal-weight default versus the down-weighted embedding. The separation's significance is the observed gap in units of that null's spread.

In [ ]:
n_seeds = 8  # a handful of independent null draws per weighting (kept modest for speed)
results = {}
for name, w in [("equal (1,1,1)", (1.0, 1.0, 1.0)), ("down-weight PLV (1,0.2,2)", (1.0, 0.2, 2.0))]:
    obs = abs(rich_qmi(ta_lo, tb_lo, w) - rich_qmi(ta_hi, tb_hi, w))
    null = np.empty(n_seeds)
    for i in range(n_seeds):
        (na, nb), (nc, nd) = matched_plv_ensembles(
            N, a1=0.4, a2_low=0.0, a2_high=0.0, seed=1000 + 10 * i
        )
        null[i] = abs(rich_qmi(na, nb, w) - rich_qmi(nc, nd, w))
    z = (obs - null.mean()) / null.std()
    results[name] = (obs, null.mean(), null.std(), z)
    print(f"{name:26s} gap={obs:.5f}  null mean={null.mean():.6f} std={null.std():.6f}  "
          f"z={z:.0f} sigma")

**Read the output.** The equal-weight embedding gives the familiar separation of a few null standard deviations — the capstone's honest, modest result. Down-weighting the PLV channel tells a different story: the gap grows *and* the null's standard deviation collapses (the matched first-moment channel was most of the noise), so the separation in sigma units jumps by orders of magnitude. The exact sigma is sensitive to the small null sample and will wobble run to run — read the *scale of the jump*, not the digits. The lesson is structural: spending the measure only where the signal lives, not where it is matched, is what makes a small effect overwhelmingly significant.

## 3. Two more levers, and the honest boundary

Weighting is the sharpest dial, but not the only one.

- **More harmonics.** A qudit $\tfrac{1}{\sqrt d}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle + e^{3i\theta}|3\rangle + \cdots\bigr)$ adds channels; the rich-QMI gap grows with $d$ even when the added moments are themselves near zero, because the measure is a global nonlinear function of the whole state.
- **A different *kind* of structure.** The amplitude embedding $\cos\alpha\,|0\rangle + \sin\alpha\,e^{i\theta}|1\rangle$ lets the $|1\rangle$ weight track signal *power*, so it can separate two systems whose phases — hence PLV — are identical but whose **phase–amplitude coupling** differs. That is a coupling PLV cannot see at all, of a different family from the higher phase moments.

We show the amplitude lever on a fresh pair: identical phases (so PLV *and* the phase qutrit are matched), but one with amplitude tied to the phase difference and one without.

In [ ]:
# Two ensembles with IDENTICAL phases (matched PLV) but different phase-amplitude coupling.
rng = np.random.default_rng(7)
theta_a = rng.uniform(0, 2 * np.pi, N)
delta = sample_phase_difference(N, a1=0.4, a2=0.0, seed=3)
theta_b = np.mod(theta_a - delta, 2 * np.pi)
amp_indep = rng.uniform(0.2, 1.0, N)                    # amplitude independent of phase
amp_coupled = 0.6 + 0.4 * np.cos(theta_a - theta_b)     # amplitude tracks the phase difference


def amp_qmi(amp_a, amp_b):
    rho = joint_density_from_states(
        amplitude_phase_state(theta_a, amp_a), amplitude_phase_state(theta_b, amp_b)
    )
    return coupling_qmi(rho, dims=(2, 2))


print(f"PLV (both ensembles): {plv(theta_a, theta_b):.4f}  (identical phases -> matched)")
q_indep = amp_qmi(amp_indep, amp_indep)
q_coupled = amp_qmi(amp_coupled, amp_coupled)
print(f"amplitude-embedding QMI: independent = {q_indep:.5f}   coupled = {q_coupled:.5f}")
print(f"  gap = {abs(q_indep - q_coupled):.5f} nats  (a channel PLV is blind to)")

In [ ]:
# The fair classical baseline: the Tort et al. (2010) modulation index. It bins the phase,
# takes the mean amplitude per bin, normalises that to a distribution P over phase bins, and
# measures its KL distance from uniform (no coupling -> uniform -> MI 0). This IS a KL
# divergence, so we compute it with the course's own kl_divergence (module 02), normalised by
# log(n_bins) to land in [0, 1]. Here the "phase" is the inter-channel phase difference delta.
def tort_modulation_index(phase, amp, n_bins=18):
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    wrapped = np.angle(np.exp(1j * phase))  # wrap to [-pi, pi)
    idx = np.clip(np.digitize(wrapped, edges) - 1, 0, n_bins - 1)
    mean_amp = np.array([amp[idx == j].mean() if np.any(idx == j) else 0.0 for j in range(n_bins)])
    p = mean_amp / mean_amp.sum()
    uniform = np.full(n_bins, 1.0 / n_bins)
    return kl_divergence(p, uniform, base=np.e) / np.log(n_bins)


delta_ab = theta_a - theta_b
mi_indep = tort_modulation_index(delta_ab, amp_indep)
mi_coupled = tort_modulation_index(delta_ab, amp_coupled)
print("classical Tort modulation index (phase = inter-channel delta, amplitude = amp):")
print(f"  independent amplitude : MI = {mi_indep:.5f}")
print(f"  coupled amplitude     : MI = {mi_coupled:.5f}")
print(f"  MI gap                : {abs(mi_indep - mi_coupled):.5f}   (the classical PAC statistic separates them too)")

**Read the two outputs together.** Both ensembles share a phase-locking value — to PLV they are the same dyad. The amplitude embedding's quantum mutual information separates them: one has amplitude riding on the phase difference, the other does not, and the amplitude channel of $\rho_{AB}$ reads that coupling directly. What it is reading is **phase–amplitude coupling (PAC)** — not a new phenomenon, but one of the most studied effects in neuroscience, where the phase of one rhythm modulates the amplitude of activity (Canolty & Knight 2010).

And — exactly as honesty demands — the classical baseline sees it too. The **Tort modulation index** (Tort et al. 2010), the standard PAC statistic, is matched (near zero) for the independent-amplitude ensemble and clearly nonzero for the coupled one: its gap separates them as plainly as the quantum measure, and arguably more directly, since it is built for this job. So the amplitude channel is *not* a quantum-exclusive detector of PAC.

**The honest boundary.** Every lever here amplifies *measurability*, never *exclusivity*. The second moment is a classical feature of the phase difference — the second-harmonic PLV $|\langle e^{2i\Delta\theta}\rangle|$ separates the `15` ensembles with a gap near $0.30$, dwarfing any rich-QMI gap. The amplitude channel reads PAC — and the Tort modulation index reads it too. What the embedding choices genuinely buy is **unification**: the *same* density-matrix coupling measure reaches phase synchrony, higher phase moments, *and* phase–amplitude coupling, each by a different embedding, where the classical toolbox needs a different hand-built statistic for each. That is a real methodological advantage, and it is the only one being claimed. Notebook `18` makes the boundary precise — and shows why no embedding, however clever, crosses it on classical data.

## Your turn

**Warm-up.** In section 1, the gap grew as we lowered $w_{h_1}$. Predict what happens in the limit $w_{h_1} \to 0$: the embedding then carries *only* the DC and second-harmonic levels. Would PLV still be recoverable from such a $\rho_{AB}$? What have you given up to maximise the second-moment sensitivity?

**Go further.** Section 2 showed the null's spread collapsing when the PLV channel is down-weighted. Explain, in terms of which channel is *matched* between the ensembles, why the first-harmonic channel contributes to the null but not to the true separation — and why removing it raises significance even more than it raises the raw gap.

**Challenge.** You want to detect **phase–amplitude** coupling *and* second-moment phase coupling at once. Sketch an embedding that carries both (hint: combine the amplitude weight of `amplitude_phase_state` with the harmonic stack of `weighted_multifreq_state`), name the two coherences of $\rho_{AB}$ that would carry each, and state the matched classical baseline you would compare against for *each* — keeping the honest accounting of `16` in view.

## What you built

You took the capstone's modest result and showed it was modest *by default*, not by necessity.

- You used `weighted_multifreq_state` to turn the embedding into a tunable instrument, and found that **down-weighting the PLV channel** enlarges the rich-QMI gap and — more importantly — collapses the null, lifting the separation's significance by orders of magnitude across the hardware noise floor.
- You saw two further levers: more harmonics, and the **amplitude embedding** opening a phase–amplitude coupling channel PLV cannot reach.
- You held the line on honesty: every lever buys **measurability**, not exclusivity — the quantum-inspired measure becomes practical to estimate, but a matched classical statistic still sees the same structure.

Take the measure of this. The embedding really is the dial of the whole framework, and turning it deliberately is the difference between an effect that hides under the noise floor and one that stands well clear of it. That is a genuine, useful gain — and you earned it without overclaiming a single thing.

## What's next

We have made the effect big. Now we ask the sharpest question of the whole arc: is there *any* embedding — even an entangling one — that lets quantum optimal transport see structure **no** classical statistic can? In `18_what_no_classical_data_embedding_can_buy` we run that experiment and meet the precise, honest boundary of the quantum-inspired program.

## References

- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments; the weighting tunes which moment the coupling measure emphasises.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, the first-moment channel we down-weight.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information, the coupling measure read off $\rho_{AB}$.
- A. B. L. Tort, R. Komorowski, H. Eichenbaum & N. Kopell, "Measuring phase-amplitude coupling between neuronal oscillations of different frequencies", *Journal of Neurophysiology* **104**, 1195–1210 (2010). DOI:10.1152/jn.00106.2010 — the modulation index, the standard classical PAC statistic and the matched baseline for the amplitude embedding here. (Metadata via PubMed.)
- R. T. Canolty & R. T. Knight, "The functional role of cross-frequency coupling", *Trends in Cognitive Sciences* **14**, 506–515 (2010). DOI:10.1016/j.tics.2010.09.001 — review establishing phase-amplitude coupling as a studied neural phenomenon. (Metadata via PubMed.)

**Previous:** `notebooks/04_QuantumOptimalTransport/16_capstone_synthesis.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/18_what_no_classical_data_embedding_can_buy.ipynb`